# Rad-Scribe Pro — Notebook 3: Model A (ResNet-50 + LSTM Baseline)
**Expected to fail** — demonstrates vanishing gradient problem with LSTM.
Establishes baseline scores for comparison with Models B, C, D.

> Run Notebook 1 first.

In [ ]:
!pip install -q datasets transformers torchvision tqdm nltk rouge-score bert-score
import nltk
nltk.download('punkt',quiet=True)
nltk.download('punkt_tab',quiet=True)
print('done')

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.3 MB/s eta 0:00:00
done


In [ ]:
import os,json,warnings
import numpy as np
import pandas as pd
import torch,torch.nn as nn
import torchvision.transforms as T
import torch.optim as optim
import matplotlib.pyplot as plt
from torch.utils.data import Dataset,DataLoader
from transformers import GPT2Tokenizer
from torchvision.models import resnet50,ResNet50_Weights
from datasets import load_dataset
from tqdm import tqdm
warnings.filterwarnings('ignore')

DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_DIR = '/content/drive/MyDrive/radscribe/radscribe/radscribe_data'
MODEL_DIR= '/content/drive/MyDrive/radscribe/radscribe/radscribe_models'
os.makedirs(MODEL_DIR,exist_ok=True)
torch.manual_seed(42)
np.random.seed(42)
print(f'Device: {DEVICE}')

Device: cuda


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
CFG = {
    'img_size'  : 224,
    'embed_size': 256,
    'hidden'    : 512,
    'num_layers': 2,
    'dropout'   : 0.3,
    'max_seq_len': 128,
    'batch_size': 32,
    'epochs'    : 10,
    'patience'  : 3,
    'lr'        : 3e-4,
    'grad_clip' : 1.0,
    'num_workers': 2,
}

tokenizer  = GPT2Tokenizer.from_pretrained('distilgpt2')
tokenizer.pad_token = tokenizer.eos_token
VOCAB_SIZE = len(tokenizer)
PAD_ID     = tokenizer.pad_token_id
EOS_ID     = tokenizer.eos_token_id
print(f'Vocab: {VOCAB_SIZE:,}')

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Vocab: 50,257


In [ ]:
PATHOLOGY_KW = [
    'pneumonia','effusion','cardiomegaly','opacity','infiltrate',
    'consolidation','atelectasis','pneumothorax','edema','fracture',
    'nodule','mass','lesion','enlarged','abnormal','disease','fibrosis'
]
def classify(r):
    t=str(r).lower()
    if any(k in t for k in PATHOLOGY_KW): return 1
    if any(w in t for w in ['normal','no significant','unremarkable','clear','no acute']): return 0
    return 2

df_train = pd.read_parquet(f'{DATA_DIR}/train.parquet')
df_val   = pd.read_parquet(f'{DATA_DIR}/val.parquet')
df_test  = pd.read_parquet(f'{DATA_DIR}/test.parquet')
df_train['split']='train'; df_val['split']='train'; df_test['split']='test'

print('Loading HF dataset...')
raw = load_dataset('MLforHealthcare/Indiana_University_Chest_X-ray_Collection')
print(f'Train {len(df_train):,} | Val {len(df_val):,} | Test {len(df_test):,}')

Loading HF dataset...


README.md:   0%|          | 0.00/463 [00:00<?, ?B/s]

data/train-00000-of-00003.parquet:   0%|          | 0.00/407M [00:00<?, ?B/s]

data/train-00001-of-00003.parquet:   0%|          | 0.00/411M [00:00<?, ?B/s]

data/train-00002-of-00003.parquet:   0%|          | 0.00/410M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/135M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/6687 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/743 [00:00<?, ? examples/s]

Train 5,683 | Val 1,004 | Test 743


In [ ]:
TRAIN_TF = T.Compose([
    T.Resize((CFG['img_size'],CFG['img_size'])),
    T.RandomHorizontalFlip(0.5),
    T.ColorJitter(brightness=0.15,contrast=0.15),
    T.ToTensor(),
    T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])
EVAL_TF = T.Compose([
    T.Resize((CFG['img_size'],CFG['img_size'])),
    T.ToTensor(),
    T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

class CXRDataset(Dataset):
    def __init__(self,df,hf_raw,transform):
        self.df=df.reset_index(drop=True)
        self.hf_raw=hf_raw
        self.transform=transform
    def __len__(self): return len(self.df)
    def __getitem__(self,idx):
        row=self.df.iloc[idx]
        img=self.hf_raw[row['split']][int(row['hf_index'])]['image'].convert('RGB')
        img=self.transform(img)
        report=str(row['report'])
        enc=tokenizer(report,max_length=CFG['max_seq_len'],
                      padding='max_length',truncation=True,return_tensors='pt')
        return {
            'image'    : img,
            'input_ids': enc['input_ids'].squeeze(0),
            'report'   : report,
            'label'    : torch.tensor(classify(report),dtype=torch.long)
        }

from torch.utils.data import WeightedRandomSampler
train_labels_arr = np.array([classify(r) for r in df_train['report']])
counts       = np.bincount(train_labels_arr,minlength=3)
class_w      = 1.0/(counts+1e-6)
sample_w     = class_w[train_labels_arr]
sampler      = WeightedRandomSampler(torch.DoubleTensor(sample_w),len(sample_w),replacement=True)

train_ds = CXRDataset(df_train,raw,TRAIN_TF)
val_ds   = CXRDataset(df_val,  raw,EVAL_TF)
test_ds  = CXRDataset(df_test, raw,EVAL_TF)
train_loader=DataLoader(train_ds,batch_size=CFG['batch_size'],sampler=sampler,
                        num_workers=CFG['num_workers'],pin_memory=True,drop_last=True)
val_loader  =DataLoader(val_ds,  batch_size=CFG['batch_size'],shuffle=False,
                        num_workers=CFG['num_workers'],pin_memory=True)
test_loader =DataLoader(test_ds, batch_size=CFG['batch_size'],shuffle=False,
                        num_workers=CFG['num_workers'],pin_memory=True)
print(f'Train {len(train_loader)} | Val {len(val_loader)} | Test {len(test_loader)} batches')

Train 177 | Val 32 | Test 24 batches


---
### Model Architecture

In [ ]:
class EncoderCNN(nn.Module):
    def __init__(self,embed_size):
        super().__init__()
        r=resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
        self.backbone=nn.Sequential(*list(r.children())[:-1])
        self.fc=nn.Linear(r.fc.in_features,embed_size)
        self.bn=nn.BatchNorm1d(embed_size,momentum=0.01)
        for name,p in self.backbone.named_parameters():
            p.requires_grad='layer4' in name
    def forward(self,x):
        return self.bn(self.fc(self.backbone(x).flatten(1)))

class DecoderRNN(nn.Module):
    def __init__(self,embed_size,hidden,vocab_size,num_layers,dropout,pad_id):
        super().__init__()
        self.embed =nn.Embedding(vocab_size,embed_size,padding_idx=pad_id)
        self.drop  =nn.Dropout(dropout)
        self.lstm  =nn.LSTM(embed_size,hidden,num_layers,batch_first=True,dropout=dropout)
        self.fc_out=nn.Linear(hidden,vocab_size)
    def forward(self,img_feat,captions):
        emb=self.drop(self.embed(captions[:,:-1]))
        inp=torch.cat([img_feat.unsqueeze(1),emb],dim=1)
        out,_=self.lstm(inp)
        return self.fc_out(out)

class ModelA(nn.Module):
    def __init__(self,cfg,vocab_size,pad_id,eos_id):
        super().__init__()
        self.eos_id =eos_id
        self.encoder=EncoderCNN(cfg['embed_size'])
        self.decoder=DecoderRNN(cfg['embed_size'],cfg['hidden'],
                                vocab_size,cfg['num_layers'],cfg['dropout'],pad_id)
    def forward(self,images,captions):
        feat=self.encoder(images)
        logits=self.decoder(feat,captions)
        labels=captions[:,1:].contiguous()
        loss=nn.CrossEntropyLoss(ignore_index=PAD_ID)(
            logits[:,:-1].reshape(-1,VOCAB_SIZE), labels.reshape(-1))
        return loss,logits
    @torch.no_grad()
    def generate(self,img_tensor,max_len=60):
        self.eval()
        img=img_tensor.unsqueeze(0).to(DEVICE)
        feat=self.encoder(img).unsqueeze(1)
        tok=torch.tensor([[self.eos_id]],device=DEVICE)
        hidden=None; gen=[]; seen_trigrams=set()
        for _ in range(max_len):
            emb=self.decoder.embed(tok)
            if hidden is None:
                out,hidden=self.decoder.lstm(torch.cat([feat,emb],dim=1))
                out=out[:,-1:,:]
            else:
                out,hidden=self.decoder.lstm(emb,hidden)
            logits=self.decoder.fc_out(out.squeeze(1))
            if len(gen)>=2:
                for pt in set(gen):
                    if (gen[-2],gen[-1],pt) in seen_trigrams:
                        logits[0,pt]-=5.0
            pred=logits.argmax(-1)
            if pred.item()==self.eos_id: break
            if len(gen)>=2:
                seen_trigrams.add((gen[-2],gen[-1],pred.item()))
            gen.append(pred.item())
            tok=pred.unsqueeze(0)
        return tokenizer.decode(gen,skip_special_tokens=True)

model_a=ModelA(CFG,VOCAB_SIZE,PAD_ID,EOS_ID).to(DEVICE)
print(f'Params: {sum(p.numel() for p in model_a.parameters()):,}')

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:01<00:00, 56.7MB/s]


Params: 66,358,929


---
### Training

In [ ]:
from transformers import get_linear_schedule_with_warmup
optimizer=optim.AdamW(filter(lambda p:p.requires_grad,model_a.parameters()),
                      lr=CFG['lr'],weight_decay=0.01)
total_steps =CFG['epochs']*len(train_loader)
warmup_steps=int(0.1*total_steps)
scheduler=get_linear_schedule_with_warmup(optimizer,warmup_steps,total_steps)
print('Ready')

Ready


In [ ]:
history={'train':[],'val':[]}
best_val=float('inf'); patience_ctr=0

for epoch in range(1,CFG['epochs']+1):
    model_a.train(); running=0.0
    pbar=tqdm(train_loader,desc=f'Epoch {epoch}/{CFG["epochs"]} [Train]')
    for batch in pbar:
        images=batch['image'].to(DEVICE)
        caps  =batch['input_ids'].to(DEVICE)
        optimizer.zero_grad()
        loss,_=model_a(images,caps)
        loss.backward()
        nn.utils.clip_grad_norm_(model_a.parameters(),CFG['grad_clip'])
        optimizer.step(); scheduler.step()
        running+=loss.item()
        pbar.set_postfix(loss=f'{loss.item():.4f}')
    train_loss=running/len(train_loader)

    model_a.eval(); val_loss=0.0
    with torch.no_grad():
        for batch in tqdm(val_loader,desc='Val',leave=False):
            loss,_=model_a(batch['image'].to(DEVICE),batch['input_ids'].to(DEVICE))
            val_loss+=loss.item()
    val_loss/=len(val_loader)
    history['train'].append(round(train_loss,4))
    history['val'  ].append(round(val_loss,4))
    print(f'Epoch {epoch}  Train: {train_loss:.4f}  Val: {val_loss:.4f}')

    if val_loss<best_val:
        best_val=val_loss; patience_ctr=0
        torch.save(model_a.state_dict(),f'{MODEL_DIR}/model_a_best.pth')
        print(f'  checkpoint saved')
    else:
        patience_ctr+=1
        if patience_ctr>=CFG['patience']:
            print(f'  early stopping at epoch {epoch}'); break

print(f'Done. Best val: {best_val:.4f}')
print('Train:',history['train'])
print('Val  :',history['val'])

Epoch 1/10 [Train]: 100%|██████████| 177/177 [01:10<00:00,  2.50it/s, loss=4.9820]


Epoch 1  Train: 7.2067  Val: 5.5134
  checkpoint saved


Epoch 2/10 [Train]: 100%|██████████| 177/177 [01:34<00:00,  1.86it/s, loss=3.4457]


Epoch 2  Train: 4.4248  Val: 4.9776
  checkpoint saved


Epoch 3/10 [Train]: 100%|██████████| 177/177 [01:22<00:00,  2.14it/s, loss=2.4864]


...

Epoch 8  Train: 1.8363  Val: 3.1167
  checkpoint saved


Epoch 9/10 [Train]: 100%|██████████| 177/177 [01:18<00:00,  2.27it/s, loss=1.8793]


Epoch 9  Train: 1.8498  Val: 3.0765
  checkpoint saved


Epoch 10/10 [Train]: 100%|██████████| 177/177 [01:17<00:00,  2.28it/s, loss=1.2861]


Epoch 10  Train: 1.8174  Val: 3.0596
  checkpoint saved
Done. Best val: 3.0596
Train: [7.2067, 4.4248, 3.4537, 2.6343, 2.2407, 2.0339, 1.9391, 1.8363, 1.8498, 1.8174]
Val  : [5.5134, 4.9776, 4.2842, 3.7464, 3.4629, 3.2973, 3.1972, 3.1167, 3.0765, 3.0596]


---
### Evaluation

In [ ]:
from nltk.translate.bleu_score import corpus_bleu,SmoothingFunction
from rouge_score import rouge_scorer as rs
from bert_score import score as bert_score

model_a.load_state_dict(torch.load(f'{MODEL_DIR}/model_a_best.pth',map_location=DEVICE))
model_a.eval()

safe_len=(df_test['hf_index']<len(raw['test'])).sum()
N=min(200,safe_len); refs,hyps=[],[]
for i in tqdm(range(N)):
    s=test_ds[i]
    g=model_a.generate(s['image'])
    refs.append(s['report'])
    hyps.append(g if g.strip() else 'no findings')

smooth=SmoothingFunction().method1
rt=[[r.lower().split()] for r in refs]
ht=[h.lower().split()   for h in hyps]
b1=corpus_bleu(rt,ht,weights=(1,0,0,0),            smoothing_function=smooth)
b4=corpus_bleu(rt,ht,weights=(.25,.25,.25,.25),     smoothing_function=smooth)
scorer=rs.RougeScorer(['rougeL'],use_stemmer=True)
rl=[scorer.score(r,h)['rougeL'] for r,h in zip(refs,hyps)]
rl_f1=sum(s.fmeasure for s in rl)/len(rl)
P,R,F1=bert_score(hyps,refs,lang='en',model_type='distilbert-base-uncased',verbose=False)
bs_f1=F1.mean().item()
print(f'BLEU-1: {b1:.4f}  BLEU-4: {b4:.4f}  ROUGE-L: {rl_f1:.4f}  BERTScore: {bs_f1:.4f}')
import json
with open(f'{MODEL_DIR}/scores_a.json','w') as f:
    json.dump({'bleu1':round(b1,4),'bleu4':round(b4,4),'rougeL_f1':round(rl_f1,4),'bertscore_f1':round(bs_f1,4)},f,indent=2)
print('Saved scores_a.json')

100%|██████████| 200/200 [00:15<00:00, 12.60it/s]


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BLEU-1: 0.0868  BLEU-4: 0.0004  ROUGE-L: 0.1432  BERTScore: 0.7574
Saved scores_a.json
